# Flight Delay Project — Visualizations for Report

All charts saved to `D:/project/outputs/charts/` as high-resolution PNG files.  
Run top to bottom — takes about 5 minutes total.


In [1]:
import pandas as pd
import numpy as np
import duckdb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os
import warnings
warnings.filterwarnings("ignore")

# ── Paths ─────────────────────────────────────────────────
JOINED_PARQUET   = "D:/project/data/processed/bts_weather_joined.parquet"
FEATURES_PARQUET = "D:/project/data/processed/features_final.parquet"
MODEL_RESULTS    = "D:/project/models/model_results.csv"
PER_AIRPORT      = "D:/project/models/per_airport_results.csv"
FEAT_IMP         = "D:/project/models/xgb_feature_importance.csv"
CHARTS_DIR       = "D:/project/outputs/charts"
os.makedirs(CHARTS_DIR, exist_ok=True)

# ── Style ──────────────────────────────────────────────────
BLUE   = "#1F4E79"
LBLUE  = "#2E75B6"
TEAL   = "#1D9E75"
AMBER  = "#BA7517"
RED    = "#A32D2D"
GREY   = "#595959"
COLORS = [BLUE, TEAL, AMBER, RED, "#534AB7", "#993556", "#3B6D11", "#D85A30", "#185FA5", "#854F0B"]

plt.rcParams.update({
    "figure.dpi":        150,
    "savefig.dpi":       200,
    "savefig.bbox":      "tight",
    "font.family":       "DejaVu Sans",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.titlesize":    14,
    "axes.labelsize":    11,
    "xtick.labelsize":   9,
    "ytick.labelsize":   9,
    "figure.facecolor":  "white",
    "axes.facecolor":    "white",
})

con = duckdb.connect()
print("Setup complete. Charts will save to:", CHARTS_DIR)


Setup complete. Charts will save to: D:/project/outputs/charts


---
## 1. Dataset Overview — Flights Per Year

In [2]:
df_year = con.execute(f"""
    SELECT
        YEAR(CAST(FL_DATE AS DATE)) AS year,
        COUNT(*) AS flights,
        ROUND(100.0 * AVG(CAST(DEP_DEL15 AS FLOAT)), 2) AS delay_rate
    FROM read_parquet('{JOINED_PARQUET}')
    WHERE AIRPORT_CODE IN (
        'ATL','LAX','ORD','DFW','DEN','JFK','SFO','CLT','LAS','PHX',
        'MIA','IAH','SEA','EWR','BOS','SLC','SAN','TPA','PDX','AUS'
    ) AND DEP_DEL15 IS NOT NULL
    GROUP BY year ORDER BY year
""").df()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
fig.suptitle("Flight Volume and Delay Rate by Year (2015–2024)", fontsize=16, fontweight="bold", color=BLUE, y=1.01)

bars = ax1.bar(df_year["year"], df_year["flights"] / 1e6, color=LBLUE, width=0.6, edgecolor="white")
ax1.set_ylabel("Flights (millions)")
ax1.set_title("Total Flights Per Year")
for bar, val in zip(bars, df_year["flights"]):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f"{val/1e6:.1f}M", ha="center", va="bottom", fontsize=8, color=GREY)

# Highlight 2020
bars[df_year["year"].tolist().index(2020)].set_color(AMBER)
ax1.annotate("COVID-19\ndrop", xy=(2020, df_year[df_year.year==2020]["flights"].values[0]/1e6),
             xytext=(2021.5, df_year["flights"].max()/1e6 * 0.7),
             arrowprops=dict(arrowstyle="->", color=AMBER), color=AMBER, fontsize=8)

ax2.plot(df_year["year"], df_year["delay_rate"], color=RED, marker="o", linewidth=2, markersize=6)
ax2.fill_between(df_year["year"], df_year["delay_rate"], alpha=0.15, color=RED)
ax2.set_ylabel("Delay Rate (%)")
ax2.set_title("Departure Delay Rate (≥15 min) Per Year")
ax2.set_xlabel("Year")
ax2.set_xticks(df_year["year"])

plt.tight_layout()
plt.savefig(f"{CHARTS_DIR}/01_flights_per_year.png")
plt.show()
print("Saved: 01_flights_per_year.png")


Saved: 01_flights_per_year.png


---
## 2. Top 10 Airports — Most Departure Delays

In [3]:
df_airport = con.execute(f"""
    SELECT
        AIRPORT_CODE,
        COUNT(*) AS total_flights,
        SUM(CASE WHEN DEP_DEL15=1 THEN 1 ELSE 0 END) AS delayed_flights,
        ROUND(100.0 * AVG(CAST(DEP_DEL15 AS FLOAT)), 2) AS delay_rate_pct,
        ROUND(AVG(DepDelayMinutes), 1) AS avg_delay_mins
    FROM read_parquet('{JOINED_PARQUET}')
    WHERE AIRPORT_CODE IN (
        'ATL','LAX','ORD','DFW','DEN','JFK','SFO','CLT','LAS','PHX',
        'MIA','IAH','SEA','EWR','BOS','SLC','SAN','TPA','PDX','AUS'
    ) AND DEP_DEL15 IS NOT NULL
    GROUP BY AIRPORT_CODE
    ORDER BY delay_rate_pct DESC
""").df()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Airport Delay Analysis — All 20 Airports", fontsize=15, fontweight="bold", color=BLUE)

# Delay rate bar chart
colors_bar = [RED if i < 5 else LBLUE for i in range(len(df_airport))]
axes[0].barh(df_airport["AIRPORT_CODE"][::-1], df_airport["delay_rate_pct"][::-1],
             color=colors_bar[::-1], edgecolor="white")
axes[0].set_xlabel("Delay Rate (%)")
axes[0].set_title("Departure Delay Rate per Airport")
for i, (rate, airport) in enumerate(zip(df_airport["delay_rate_pct"][::-1], df_airport["AIRPORT_CODE"][::-1])):
    axes[0].text(rate + 0.2, i, f"{rate:.1f}%", va="center", fontsize=8, color=GREY)
axes[0].set_xlim(0, df_airport["delay_rate_pct"].max() * 1.15)

# Total flights bar chart
df_sorted_vol = df_airport.sort_values("total_flights", ascending=True)
axes[1].barh(df_sorted_vol["AIRPORT_CODE"], df_sorted_vol["total_flights"] / 1e6,
             color=TEAL, edgecolor="white")
axes[1].set_xlabel("Total Flights (millions)")
axes[1].set_title("Total Flight Volume per Airport")
for i, (vol, airport) in enumerate(zip(df_sorted_vol["total_flights"], df_sorted_vol["AIRPORT_CODE"])):
    axes[1].text(vol/1e6 + 0.02, i, f"{vol/1e6:.1f}M", va="center", fontsize=8, color=GREY)

plt.tight_layout()
plt.savefig(f"{CHARTS_DIR}/02_airport_delay_analysis.png")
plt.show()
print("Saved: 02_airport_delay_analysis.png")


Saved: 02_airport_delay_analysis.png


---
## 3. Delay Rate by Month — Seasonality

In [4]:
df_month = con.execute(f"""
    SELECT
        MONTH(CAST(FL_DATE AS DATE)) AS month,
        COUNT(*) AS flights,
        ROUND(100.0 * AVG(CAST(DEP_DEL15 AS FLOAT)), 2) AS delay_rate,
        ROUND(AVG(DepDelayMinutes), 1) AS avg_delay_mins
    FROM read_parquet('{JOINED_PARQUET}')
    WHERE AIRPORT_CODE IN (
        'ATL','LAX','ORD','DFW','DEN','JFK','SFO','CLT','LAS','PHX',
        'MIA','IAH','SEA','EWR','BOS','SLC','SAN','TPA','PDX','AUS'
    ) AND DEP_DEL15 IS NOT NULL
    GROUP BY month ORDER BY month
""").df()

months = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
df_month["month_name"] = [months[m-1] for m in df_month["month"]]

fig, ax = plt.subplots(figsize=(12, 5))
bar_colors = [RED if r > df_month["delay_rate"].mean() + df_month["delay_rate"].std()
              else AMBER if r > df_month["delay_rate"].mean()
              else TEAL for r in df_month["delay_rate"]]
bars = ax.bar(df_month["month_name"], df_month["delay_rate"], color=bar_colors, edgecolor="white", width=0.6)
ax.axhline(df_month["delay_rate"].mean(), color=GREY, linestyle="--", linewidth=1, alpha=0.7, label="Average")
ax.set_title("Departure Delay Rate by Month (2015–2024)", fontsize=14, fontweight="bold", color=BLUE)
ax.set_ylabel("Delay Rate (%)")
ax.set_xlabel("Month")
for bar, rate in zip(bars, df_month["delay_rate"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f"{rate:.1f}%", ha="center", va="bottom", fontsize=8, color=GREY)
ax.legend()
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=RED, label="High delay season"),
                   Patch(facecolor=AMBER, label="Above average"),
                   Patch(facecolor=TEAL, label="Below average")]
ax.legend(handles=legend_elements, loc="upper right")
plt.tight_layout()
plt.savefig(f"{CHARTS_DIR}/03_delay_by_month.png")
plt.show()
print("Saved: 03_delay_by_month.png")


Saved: 03_delay_by_month.png


---
## 4. Delay Causes Breakdown

In [5]:
df_causes = con.execute(f"""
    SELECT
        ROUND(AVG(CASE WHEN CarrierDelay > 0 THEN CarrierDelay END), 1) AS carrier,
        ROUND(AVG(CASE WHEN WeatherDelay > 0 THEN WeatherDelay END), 1) AS weather,
        ROUND(AVG(CASE WHEN NASDelay > 0 THEN NASDelay END), 1) AS nas,
        ROUND(AVG(CASE WHEN LateAircraftDelay > 0 THEN LateAircraftDelay END), 1) AS late_aircraft,
        ROUND(AVG(CASE WHEN SecurityDelay > 0 THEN SecurityDelay END), 1) AS security
    FROM read_parquet('{JOINED_PARQUET}')
    WHERE DEP_DEL15 = 1
    AND AIRPORT_CODE IN (
        'ATL','LAX','ORD','DFW','DEN','JFK','SFO','CLT','LAS','PHX',
        'MIA','IAH','SEA','EWR','BOS','SLC','SAN','TPA','PDX','AUS'
    )
""").df()

causes = ["Carrier", "Weather", "NAS / ATC", "Late Aircraft", "Security"]
values = [df_causes["carrier"].values[0], df_causes["weather"].values[0],
          df_causes["nas"].values[0], df_causes["late_aircraft"].values[0],
          df_causes["security"].values[0]]
cause_colors = [LBLUE, TEAL, AMBER, RED, "#534AB7"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))
fig.suptitle("Delay Cause Analysis (Delayed Flights Only)", fontsize=14, fontweight="bold", color=BLUE)

ax1.barh(causes[::-1], values[::-1], color=cause_colors[::-1], edgecolor="white")
ax1.set_xlabel("Average Delay Minutes")
ax1.set_title("Avg Minutes per Delay Cause")
for i, v in enumerate(values[::-1]):
    ax1.text(v + 0.3, i, f"{v:.0f} min", va="center", fontsize=9, color=GREY)

wedges, texts, autotexts = ax2.pie(values, labels=causes, colors=cause_colors,
                                    autopct="%1.1f%%", startangle=140,
                                    wedgeprops={"edgecolor": "white", "linewidth": 1.5})
for t in autotexts:
    t.set_fontsize(9)
ax2.set_title("Share of Delay Causes")

plt.tight_layout()
plt.savefig(f"{CHARTS_DIR}/04_delay_causes.png")
plt.show()
print("Saved: 04_delay_causes.png")


Saved: 04_delay_causes.png


---
## 5. Weather Severity vs Delay Rate

In [6]:
df_weather = con.execute(f"""
    SELECT
        weather_severity_score,
        COUNT(*) AS flights,
        ROUND(100.0 * AVG(CAST(DEP_DEL15 AS FLOAT)), 2) AS delay_rate,
        ROUND(AVG(DepDelayMinutes), 1) AS avg_delay_mins
    FROM read_parquet('{FEATURES_PARQUET}')
    WHERE DEP_DEL15 IS NOT NULL
    AND weather_severity_score IS NOT NULL
    GROUP BY weather_severity_score
    ORDER BY weather_severity_score
""").df()

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

score_labels = ["0\n(Clear)", "1\n(Minor)", "2\n(Moderate)", "3\n(Significant)", "4\n(Severe)", "5\n(Extreme)"]
x = range(len(df_weather))

bars = ax1.bar(x, df_weather["flights"] / 1e6, color=LBLUE, alpha=0.6,
               width=0.5, label="Flight volume (M)", zorder=2)
line = ax2.plot(x, df_weather["delay_rate"], color=RED, marker="o",
                linewidth=2.5, markersize=8, label="Delay rate (%)", zorder=3)

ax1.set_xticks(x)
ax1.set_xticklabels(score_labels[:len(df_weather)])
ax1.set_xlabel("Weather Severity Score (0=clear, 5=extreme)")
ax1.set_ylabel("Flights (millions)", color=LBLUE)
ax2.set_ylabel("Delay Rate (%)", color=RED)
ax1.set_title("Weather Severity Score vs Flight Delay Rate", fontsize=14, fontweight="bold", color=BLUE)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.tight_layout()
plt.savefig(f"{CHARTS_DIR}/05_weather_severity_vs_delay.png")
plt.show()
print("Saved: 05_weather_severity_vs_delay.png")


Saved: 05_weather_severity_vs_delay.png


---
## 6. Delay Rate by Day of Week and Time of Day

In [7]:
df_dow = con.execute(f"""
    SELECT day_of_week,
           ROUND(100.0 * AVG(CAST(DEP_DEL15 AS FLOAT)), 2) AS delay_rate
    FROM read_parquet('{FEATURES_PARQUET}')
    WHERE DEP_DEL15 IS NOT NULL
    GROUP BY day_of_week ORDER BY day_of_week
""").df()

df_tod = con.execute(f"""
    SELECT time_of_day,
           ROUND(100.0 * AVG(CAST(DEP_DEL15 AS FLOAT)), 2) AS delay_rate,
           COUNT(*) AS flights
    FROM read_parquet('{FEATURES_PARQUET}')
    WHERE DEP_DEL15 IS NOT NULL AND time_of_day IS NOT NULL AND time_of_day != 'nan'
    GROUP BY time_of_day
""").df()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Delay Rate by Day of Week and Time of Day", fontsize=14, fontweight="bold", color=BLUE)

days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
dow_colors = [RED if d in ["Thu", "Fri"] else TEAL if d in ["Tue", "Sat"] else LBLUE for d in days]
ax1.bar(days, df_dow["delay_rate"], color=dow_colors, edgecolor="white", width=0.6)
ax1.set_ylabel("Delay Rate (%)")
ax1.set_title("By Day of Week")
ax1.axhline(df_dow["delay_rate"].mean(), color=GREY, linestyle="--", linewidth=1, alpha=0.7)
for i, rate in enumerate(df_dow["delay_rate"]):
    ax1.text(i, rate + 0.1, f"{rate:.1f}%", ha="center", va="bottom", fontsize=8, color=GREY)

tod_order = ["RedEye", "Morning", "Afternoon", "Evening", "Night"]
tod_colors_map = {"RedEye": "#534AB7", "Morning": TEAL, "Afternoon": AMBER,
                  "Evening": RED, "Night": BLUE}
df_tod_ordered = df_tod.set_index("time_of_day").reindex(
    [t for t in tod_order if t in df_tod["time_of_day"].values]
).reset_index()
tod_bar_colors = [tod_colors_map.get(t, LBLUE) for t in df_tod_ordered["time_of_day"]]
ax2.bar(df_tod_ordered["time_of_day"], df_tod_ordered["delay_rate"],
        color=tod_bar_colors, edgecolor="white", width=0.6)
ax2.set_ylabel("Delay Rate (%)")
ax2.set_title("By Time of Day")
for i, rate in enumerate(df_tod_ordered["delay_rate"]):
    ax2.text(i, rate + 0.1, f"{rate:.1f}%", ha="center", va="bottom", fontsize=8, color=GREY)

plt.tight_layout()
plt.savefig(f"{CHARTS_DIR}/06_delay_by_dow_tod.png")
plt.show()
print("Saved: 06_delay_by_dow_tod.png")


Saved: 06_delay_by_dow_tod.png


---
## 7. Hub vs Non-Hub Airport Comparison

In [8]:
df_hub = con.execute(f"""
    SELECT
        is_hub,
        COUNT(*) AS flights,
        ROUND(100.0 * AVG(CAST(DEP_DEL15 AS FLOAT)), 2) AS delay_rate,
        ROUND(AVG(DepDelayMinutes), 1) AS avg_delay_mins,
        ROUND(AVG(congestion_index), 3) AS avg_congestion
    FROM read_parquet('{FEATURES_PARQUET}')
    WHERE DEP_DEL15 IS NOT NULL
    GROUP BY is_hub
""").df()

df_carrier = con.execute(f"""
    SELECT
        Reporting_Airline AS airline,
        COUNT(*) AS flights,
        ROUND(100.0 * AVG(CAST(DEP_DEL15 AS FLOAT)), 2) AS delay_rate
    FROM read_parquet('{FEATURES_PARQUET}')
    WHERE DEP_DEL15 IS NOT NULL
    GROUP BY Reporting_Airline
    ORDER BY delay_rate DESC
    LIMIT 12
""").df()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Hub vs Non-Hub and Carrier Delay Analysis", fontsize=14, fontweight="bold", color=BLUE)

hub_labels = ["Non-Hub", "Hub"]
hub_colors = [TEAL, LBLUE]
metrics = ["delay_rate", "avg_congestion"]
x = np.arange(len(hub_labels))
width = 0.35
bars1 = ax1.bar(x - width/2, df_hub.sort_values("is_hub")["delay_rate"],
                width, color=LBLUE, label="Delay Rate (%)", edgecolor="white")
ax1_r = ax1.twinx()
bars2 = ax1_r.bar(x + width/2, df_hub.sort_values("is_hub")["avg_congestion"],
                   width, color=AMBER, label="Congestion Index", edgecolor="white")
ax1.set_xticks(x)
ax1.set_xticklabels(hub_labels)
ax1.set_ylabel("Delay Rate (%)", color=LBLUE)
ax1_r.set_ylabel("Congestion Index", color=AMBER)
ax1.set_title("Hub vs Non-Hub Airports")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1_r.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

carrier_colors = [RED if r > df_carrier["delay_rate"].quantile(0.75)
                  else TEAL if r < df_carrier["delay_rate"].quantile(0.25)
                  else LBLUE for r in df_carrier["delay_rate"]]
ax2.barh(df_carrier["airline"][::-1], df_carrier["delay_rate"][::-1],
         color=carrier_colors[::-1], edgecolor="white")
ax2.set_xlabel("Delay Rate (%)")
ax2.set_title("Delay Rate by Carrier")
for i, rate in enumerate(df_carrier["delay_rate"][::-1]):
    ax2.text(rate + 0.1, i, f"{rate:.1f}%", va="center", fontsize=8, color=GREY)

plt.tight_layout()
plt.savefig(f"{CHARTS_DIR}/07_hub_carrier_analysis.png")
plt.show()
print("Saved: 07_hub_carrier_analysis.png")


Saved: 07_hub_carrier_analysis.png


---
## 8. Model Performance Comparison

In [9]:
df_models = pd.read_csv(MODEL_RESULTS)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Model Performance Comparison — Test Set (2022–2024)",
             fontsize=14, fontweight="bold", color=BLUE)

metrics = ["auc", "f1", "precision", "recall"]
metric_labels = ["AUC-ROC", "F1 Score", "Precision", "Recall"]
model_colors = [LBLUE, TEAL, AMBER, RED]

x = np.arange(len(metrics))
width = 0.8 / len(df_models)
for i, (_, row) in enumerate(df_models.iterrows()):
    vals = [row[m] for m in metrics]
    offset = (i - len(df_models)/2 + 0.5) * width
    bars = axes[0].bar(x + offset, vals, width * 0.9,
                       label=row["model"].split("(")[0].strip(),
                       color=model_colors[i % len(model_colors)],
                       edgecolor="white")

axes[0].set_xticks(x)
axes[0].set_xticklabels(metric_labels)
axes[0].set_ylabel("Score")
axes[0].set_ylim(0, 1.0)
axes[0].set_title("All Metrics by Model")
axes[0].legend(fontsize=8)
axes[0].axhline(0.5, color=GREY, linestyle="--", linewidth=0.8, alpha=0.5)

best = df_models.sort_values("auc", ascending=False).iloc[0]
remaining = df_models[df_models["model"] != best["model"]]

categories = ["AUC-ROC", "F1", "Precision", "Recall"]
best_vals = [best["auc"], best["f1"], best["precision"], best["recall"]]
axes[1].barh(categories[::-1], best_vals[::-1],
             color=[TEAL, LBLUE, AMBER, RED][::-1], edgecolor="white")
axes[1].set_xlim(0, 1.0)
axes[1].set_title(f"Best Model: {best['model'].split('(')[0].strip()}")
axes[1].axvline(0.5, color=GREY, linestyle="--", linewidth=0.8, alpha=0.5)
for i, v in enumerate(best_vals[::-1]):
    axes[1].text(v + 0.01, i, f"{v:.4f}", va="center", fontsize=9, color=GREY)

plt.tight_layout()
plt.savefig(f"{CHARTS_DIR}/08_model_comparison.png")
plt.show()
print("Saved: 08_model_comparison.png")


Saved: 08_model_comparison.png


---
## 9. Per-Airport Model Accuracy

In [10]:
df_pa = pd.read_csv(PER_AIRPORT)

fig, ax = plt.subplots(figsize=(12, 7))
x = np.arange(len(df_pa))
width = 0.35

bars1 = ax.bar(x - width/2, df_pa["actual_delay_pct"], width,
               label="Actual delay %", color=RED, edgecolor="white", alpha=0.85)
bars2 = ax.bar(x + width/2, df_pa["accuracy_pct"], width,
               label="Model accuracy %", color=TEAL, edgecolor="white", alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(df_pa["AIRPORT_CODE"], rotation=45, ha="right")
ax.set_ylabel("Percentage (%)")
ax.set_title("Per-Airport: Actual Delay Rate vs Model Accuracy (Test 2022–2024)",
             fontsize=13, fontweight="bold", color=BLUE)
ax.legend()
ax.set_ylim(0, 100)
ax.axhline(df_pa["accuracy_pct"].mean(), color=GREY, linestyle="--",
           linewidth=1, alpha=0.7, label="Avg accuracy")

plt.tight_layout()
plt.savefig(f"{CHARTS_DIR}/09_per_airport_accuracy.png")
plt.show()
print("Saved: 09_per_airport_accuracy.png")


Saved: 09_per_airport_accuracy.png


---
## 10. Feature Importance (XGBoost)

In [11]:
try:
    df_fi = pd.read_csv(FEAT_IMP).head(20)
    fig, ax = plt.subplots(figsize=(11, 8))
    colors_fi = [RED if i < 5 else LBLUE if i < 10 else TEAL
                 for i in range(len(df_fi))]
    ax.barh(df_fi["feature"][::-1], df_fi["importance"][::-1],
            color=colors_fi[::-1], edgecolor="white")
    ax.set_xlabel("Importance Score (XGBoost Gain)")
    ax.set_title("Top 20 Feature Importances — XGBoost Model",
                 fontsize=14, fontweight="bold", color=BLUE)
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=RED, label="Top 5 features"),
                       Patch(facecolor=LBLUE, label="Features 6–10"),
                       Patch(facecolor=TEAL, label="Features 11–20")]
    ax.legend(handles=legend_elements, loc="lower right")
    plt.tight_layout()
    plt.savefig(f"{CHARTS_DIR}/10_feature_importance.png")
    plt.show()
    print("Saved: 10_feature_importance.png")
except FileNotFoundError:
    print("xgb_feature_importance.csv not found — skipping")


Saved: 10_feature_importance.png


---
## 11. Delay Risk Score Distribution

In [12]:
try:
    df_risk = pd.read_csv("D:/project/models/risk_score_distribution.csv")
    fig, ax = plt.subplots(figsize=(10, 5))
    risk_colors = [TEAL, "#3B6D11", AMBER, "#D85A30", RED]
    bars = ax.bar(df_risk["risk_band"], df_risk["actual_delay_pct"],
                  color=risk_colors[:len(df_risk)], edgecolor="white", width=0.6)
    ax2 = ax.twinx()
    ax2.plot(range(len(df_risk)), df_risk["flights"] / 1e6,
             color=BLUE, marker="o", linewidth=2, markersize=7, label="Flights (M)")
    ax.set_ylabel("Actual Delay Rate (%)", color=RED)
    ax2.set_ylabel("Flight Volume (millions)", color=BLUE)
    ax.set_title("Travel Risk Score Bands — Calibration",
                 fontsize=14, fontweight="bold", color=BLUE)
    ax.set_xlabel("Risk Band")
    for bar, rate in zip(bars, df_risk["actual_delay_pct"]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f"{rate:.1f}%", ha="center", va="bottom", fontsize=9, color=GREY)
    plt.tight_layout()
    plt.savefig(f"{CHARTS_DIR}/11_risk_score_distribution.png")
    plt.show()
    print("Saved: 11_risk_score_distribution.png")
except FileNotFoundError:
    print("risk_score_distribution.csv not found — skipping")


Saved: 11_risk_score_distribution.png


---
## 12. COVID Impact — 2020 vs Other Years

In [14]:
df_covid = con.execute(f"""
    SELECT
        YEAR(CAST(FL_DATE AS DATE)) AS year,
        COUNT(*) AS flights,
        ROUND(100.0 * AVG(CAST(DEP_DEL15 AS FLOAT)), 2) AS delay_rate,
        ROUND(AVG(DepDelayMinutes), 1) AS avg_delay_mins
    FROM read_parquet('{FEATURES_PARQUET}')
    WHERE DEP_DEL15 IS NOT NULL
    GROUP BY YEAR(CAST(FL_DATE AS DATE))
    ORDER BY year
""").df()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("COVID-19 Impact on Flight Operations (2020)",
             fontsize=14, fontweight="bold", color=BLUE)

bar_colors = [AMBER if y == 2020 else LBLUE for y in df_covid["year"]]
axes[0].bar(df_covid["year"], df_covid["flights"] / 1e6,
            color=bar_colors, edgecolor="white", width=0.6)
axes[0].set_title("Flight Volume — 2020 vs Other Years")
axes[0].set_ylabel("Flights (millions)")
axes[0].set_xlabel("Year")
axes[0].annotate("COVID\n-60% volume", xy=(2020, df_covid[df_covid.year==2020]["flights"].values[0]/1e6),
                  xytext=(2022, df_covid["flights"].max()/1e6 * 0.6),
                  arrowprops=dict(arrowstyle="->", color=AMBER), color=AMBER, fontsize=9)

axes[1].plot(df_covid["year"], df_covid["delay_rate"],
             marker="o", color=RED, linewidth=2, markersize=7)
axes[1].fill_between(df_covid["year"], df_covid["delay_rate"], alpha=0.1, color=RED)
axes[1].set_title("Delay Rate — Paradox: Low in 2020")
axes[1].set_ylabel("Delay Rate (%)")
axes[1].set_xlabel("Year")
axes[1].annotate("Paradoxically low\n(empty aircraft,\nno congestion)",
                  xy=(2020, df_covid[df_covid.year==2020]["delay_rate"].values[0]),
                  xytext=(2017, df_covid["delay_rate"].max() * 0.9),
                  arrowprops=dict(arrowstyle="->", color=AMBER), color=AMBER, fontsize=8)

plt.tight_layout()
plt.savefig(f"{CHARTS_DIR}/12_covid_impact.png")
plt.show()
print("Saved: 12_covid_impact.png")


Saved: 12_covid_impact.png


In [15]:
con.close()

import glob
chart_files = sorted(glob.glob(f"{CHARTS_DIR}/*.png"))
print(f"\n{'='*55}")
print(f"  ALL CHARTS COMPLETE")
print(f"{'='*55}")
print(f"  Total charts saved : {len(chart_files)}")
print(f"  Location           : {CHARTS_DIR}")
print()
for f in chart_files:
    size_kb = os.path.getsize(f) / 1024
    print(f"  {os.path.basename(f):<45} {size_kb:.0f} KB")
print(f"{'='*55}")
print("  Copy the charts folder into your report document.")



  ALL CHARTS COMPLETE
  Total charts saved : 12
  Location           : D:/project/outputs/charts

  01_flights_per_year.png                       123 KB
  02_airport_delay_analysis.png                 146 KB
  03_delay_by_month.png                         77 KB
  04_delay_causes.png                           135 KB
  05_weather_severity_vs_delay.png              106 KB
  06_delay_by_dow_tod.png                       84 KB
  07_hub_carrier_analysis.png                   114 KB
  08_model_comparison.png                       95 KB
  09_per_airport_accuracy.png                   84 KB
  10_feature_importance.png                     126 KB
  11_risk_score_distribution.png                105 KB
  12_covid_impact.png                           115 KB
  Copy the charts folder into your report document.
